<img src="Fink_PrimaryLogo_WEB.jpg" width=400 />

# Use case: Manipulating outputs from the Data Transfer Service

- From tutorial : https://github.com/astrolabsoftware/fink-tutorials/tree/main/lsst

The [Fink data transfer service](https://lsst.fink-portal.org/download) was developed due to the need to transfer large quantities of data for statistical and population studies, as well as to enable training of machine learning models (see [Fraga *et al*, 2024](https://www.aanda.org/articles/aa/full_html/2024/12/aa50370-24/aa50370-24.html)). 

The service has been extensively used during its operation with ZTF data. 

For LSST, we continue to provide the community with easy access to large volumes of data, however, the post-processing of the results requires some fine tunning, given the complexity of the survey. 

You can discover details on how to fetch data from Fink in the documentation. 

In this notebook, we provide you with a few examples on how to manipulate the resulting parquet files. 

## Environment set up

In order to run this notebook, you will need to install the [fink-client](https://github.com/astrolabsoftware/fink-client) and [pyarrow](https://pypi.org/project/pyarrow/). 

Then you can import the following libraries

In [ ]:
import numpy as np
import glob
import matplotlib.pylab as plt
import matplotlib
import pyarrow.parquet as pq

from fink_client.visualisation import extract_field
from fink_client.visualisation import show_stamps

# For plots
font = {"weight": "bold", "size": 16}

matplotlib.rc("font", **font)

### Case 1: Reading the data

Suppose that the result of your query to the data transfer service is store in a directory named ... 

In [ ]:
# topicname = "ftransfer_lsst_2026-03-27_750709"
topicname = "ftransfer_lsst_2026-05-06_481298"

In [ ]:
# Example using pyarrow Table
schema = pq.read_schema(f"arrow_schema_{topicname}.metadata")
table = pq.read_table(topicname, schema=schema)

You could easily get a pandas dataframe for convenience, although beware of cast for `diaObjectId` (see https://doc.lsst.fink-broker.org/services/data_transfer/#diaobjectid-casting-errors)

In [ ]:
# Make a list of dict for easier manipulation
alerts = table.to_pylist()

### Case 2: accessing alert history

Now that your data is available as dictionaries, you are free to explore the potential of the alert package. Important things to keep in mind:

1.  An alert is one point --> `diaSource`
2.  The alert package contains the previous photometric history of that position --> `prvDiaSources` ...
3.  Starting from the second alert, it also contains 30 days of forced photometry preceeding the first alert --> `prvDiaForcedSources`

You can use any information contained in these fields for the construction of your model or filter.

Let's illustrate this task by separating 1 alert:

In [ ]:
# Get one alert
alert = alerts[0]

In [ ]:
alert.keys()

You can explore the meaning of all fields in the schema page: https://lsst.fink-portal.org/schemas. In order to reconstruct the light curve from this, we will use the function below

Let's use it to gather the entire light curve information in a single data frame

In [ ]:
mjds = extract_field(
    alert, "midpointMjdTai", current="diaSource", previous="prvDiaSources"
)  # get observation days (MJD)
psfFlux = extract_field(
    alert, "psfFlux", current="diaSource", previous="prvDiaSources"
)  # get difference fluxes
psfFluxErr = extract_field(
    alert, "psfFluxErr", current="diaSource", previous="prvDiaSources"
)  # get difference flux errors
bands = extract_field(alert, "band", current="diaSource", previous="prvDiaSources")  # get filter bands

We can now plot the entire photometric history for this source

In [ ]:
# all lsst bands
all_bands = ["u", "g", "r", "i", "z", "y"]
symbols = ["o", "<", ">", "s", "*", "p"]

# get unique filters
lc_bands = np.unique(bands)

plt.figure(figsize=(10, 6))

for i in range(len(all_bands)):
    if all_bands[i] in lc_bands:
        # identify one filter at a time
        flag = bands == all_bands[i]
        plt.errorbar(mjds[flag], psfFlux[flag], yerr=psfFluxErr[flag], fmt=symbols[i], label=all_bands[i])

plt.xlabel("MJD")
plt.ylabel("difference flux (nJy)")
plt.legend(loc="upper left")
plt.show()

### Case 3: accessing alert stamps

In [ ]:
fig = plt.figure()
show_stamps(alert, fig)